### Import Dependencies

In [1]:
import openai
import json
from dotenv import load_dotenv

from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue

from langsmith import Client

load_dotenv("../../.env", override=True)

True

### Download all data from Qdrant

In [2]:
qdrant_client = QdrantClient(url="http://localhost:6333")

all_points = qdrant_client.scroll(
    collection_name="Amazon-items-collection-01",
    limit=100,
    offset=None,
    with_payload=True,
    with_vectors=False,
)

all_context = [
    {"id": data.payload["parent_asin"], "text": data.payload["preprocessed_description"]}
    for data in all_points[0]
]
len(all_context)

50

### Render a prompt to generate synthetic Eval reference dataset

In [3]:
output_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "reasoning": {
                "type": "string",
                "description": "Reasoning why the question could be answered with the chunks.",
            },
            "question": {
                "type": "string",
                "description": "Suggested question.",
            },
            "chunk_ids": {
                "type": "array",
                "items": {
                    "type": "string",
                    "description": "ID of the chunk that could be used to answer the question.",
                },
            },
            "answer_example": {
                "type": "string",
                "description": "Suggested answer grounded in the context.",
            },
        },
    },
}

SYSTEM_PROMPT = f"""
I am building a RAG application. I have a collection of 50 chunks of text.
The RAG application will act as a shopping assistant that can answer questions about the stock of the products we have available.
I will provide all of the available products to you with IDs of each chunk.
Come up with 30 questions to which the answers could be grounded in the chunk context.
The questions should imitate a potential real user of this RAG system - a customer of the e-shop.
As an output I need you to provide me the list of questions and the IDs of the chunks that could be used to answer them.
Also, provide an example answer to the question given the context of the chunks.
Also, provide the reason why you chose the chunks to answer the questions.
Construct 10 questions that could use multiple chunks in the answer.
Construct 15 questions that could use single chunk in the answer.
Construct 5 questions that can't be answered with the available chunks.
Don't use word "chunks" in suggested questions, refer to the chunks as products.

<OUTPUT JSON SCHEMA>
{json.dumps(output_schema, indent=2)}
</OUTPUT JSON SCHEMA>

I need to be able to parse the json output.
"""

USER_PROMPT = f"""
Here is the list of chunks, each list element is a dictionary with id and text:
{json.dumps(all_context, indent=2)}
"""

### Generate the questions

In [4]:
response = openai.chat.completions.create(
    model="gpt-5.5",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ],
    reasoning_effort="none",
)
response.usage

CompletionUsage(completion_tokens=6037, prompt_tokens=19784, total_tokens=25821, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None))

In [5]:
# The model can wrap the JSON in a ```json code block, so remove it before parsing
raw_output = response.choices[0].message.content.strip()
if raw_output.startswith("```"):
    raw_output = raw_output.split("\n", 1)[1].rsplit("```", 1)[0]

json_output = json.loads(raw_output)
len(json_output)

34

In [6]:
single_chunk_counter = 0
multiple_chunk_counter = 0
impossible_counter = 0

for item in json_output:
    if len(item["chunk_ids"]) == 1:
        single_chunk_counter += 1
    elif len(item["chunk_ids"]) > 1:
        multiple_chunk_counter += 1
    else:
        impossible_counter += 1

print(f"Single chunk questions: {single_chunk_counter}")
print(f"Multiple chunk questions: {multiple_chunk_counter}")
print(f"Impossible questions: {impossible_counter}")

Single chunk questions: 19
Multiple chunk questions: 10
Impossible questions: 5


In [7]:
# The model can invent product IDs, so check every ID against the real products
valid_ids = {item["id"] for item in all_context}
invalid_ids = [
    (item["question"], chunk_id)
    for item in json_output
    for chunk_id in item["chunk_ids"]
    if chunk_id not in valid_ids
]
print(f"Invalid IDs: {len(invalid_ids)}")
invalid_ids

Invalid IDs: 0


[]

### Look up product descriptions by ID

In [8]:
def get_description(parent_asin: str) -> str:
    points = qdrant_client.scroll(
        collection_name="Amazon-items-collection-01",
        scroll_filter=Filter(
            must=[
                FieldCondition(
                    key="parent_asin",
                    match=MatchValue(value=parent_asin),
                )
            ]
        ),
    )[0]

    return points[0].payload["preprocessed_description"]

get_description("B0BRJS644Z")

'Marame 120mm 5v USB Powered Fan with Speed Controller Cooling for Router Modem Receiver DVR Xbox TV Box (120mm x 120mm x 55mm) 【Solve Your Cooling Problem】\xa0This fan will do the job keeping your devices from overheating and keep your electronics running cool. You can also use them in confined spaces for cooling various electronics. blowing cool air through it to aid in longevity. 【Speed\xa0Control\xa0Switch】 The speed controller located on the cord allows you to adjust the fan’s speed from off to low, medium, and high. This enables you to set the fan to optimal noise and airflow levels for various environments. 【High Compatibility USB-Powered】 Powered by a 3.3ft USB cable. Compatible with desktop, laptop, power bank, AC adapters, car chargers, and other power supplies that support USB connection. USB fan is energy-saving and environmentally friendly. 【Rubber Feet & Dust Filter】\xa0Rubber Feet on the fan raise it up enough off of the surface it is sitting on to allow enough replaceme

### Create Eval dataset in LangSmith

In [9]:
ls_client = Client()
dataset_name = "rag-evaluation-dataset"

# Running this cell again must not add the same examples a second time
if ls_client.has_dataset(dataset_name=dataset_name):
    dataset = ls_client.read_dataset(dataset_name=dataset_name)
    print(f"Dataset already exists with {dataset.example_count} examples, skipping upload.")
else:
    dataset = ls_client.create_dataset(
        dataset_name=dataset_name,
        description="RAG evaluation dataset",
    )
    for item in json_output:
        chunk_ids = [chunk_id for chunk_id in item["chunk_ids"] if chunk_id in valid_ids]
        ls_client.create_example(
            dataset_id=dataset.id,
            inputs={
                "question": item["question"],
            },
            outputs={
                "ground_truth": item["answer_example"],
                "reference_context_ids": chunk_ids,
                "reference_descriptions": [get_description(chunk_id) for chunk_id in chunk_ids],
            },
        )
    print(f"Uploaded {len(json_output)} examples to {dataset_name}.")

Uploaded 34 examples to rag-evaluation-dataset.
